<a href="https://colab.research.google.com/github/Niraj-Bhatta/RAG-using-langchain/blob/main/rag_yt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Rag project

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

from google.colab import userdata
api_key = userdata.get('OpenAIKey')


In [3]:
%pip install -q youtube-transcript-api langchain-community langchain-openai faiss-cpu tiktoken python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
!pip install -q langchain-huggingface sentence-transformers langchain-community faiss-cpu

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

In [12]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

# Using HuggingFace exclusively instead of OpenAI
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

#step 1a -indexing(Document INgestion)

In [32]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "GtaxU6DZvLs"

try:
    # Fetch YouTube transcript
    transcript_data = YouTubeTranscriptApi().fetch(
        video_id,
        languages=["en"]
    )

    # Convert transcript into a single string
    transcript = " ".join(
        chunk.text for chunk in transcript_data
    )

    print("Successfully fetched transcript!")
    print(transcript[:500] + "...")

except Exception as e:
    print("Transcript retrieval failed!")
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

    transcript = None

Transcript retrieval failed!
Error type: RequestBlocked
Error details: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=GtaxU6DZvLs! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventuall

#step 1b indexing(text splitting)

In [15]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""])
chunks = splitter.create_documents([transcript])

In [16]:
len(chunks)

1

In [17]:
chunks[0]

Document(metadata={}, page_content='Elon Musk is a technology entrepreneur and business magnate. He is the founder, CEO, and chief engineer of SpaceX, and an early-stage investor, CEO, and product architect of Tesla, Inc. In public speeches, Elon Musk has frequently discussed the future of artificial intelligence, space exploration, and sustainable energy. He believes that AI needs careful regulation to ensure it remains safe for humanity, and he envisions colonizing Mars to make life multiplanetary.')

#step 1c and 1d

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
# Check the number of stored vectors
print("Stored vectors:", vector_store.index.ntotal)

# Check the embedding dimensions
print("Embedding dimensions:", vector_store.index.d)

# Check the number of document chunks
print("Document chunks:", len(vector_store.docstore._dict))

Stored vectors: 1
Embedding dimensions: 384
Document chunks: 1


##Step 2 Retrievel

In [22]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})


In [23]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7a94935f4980>, search_kwargs={'k': 4})

In [24]:
retriever.invoke(' What are Elons sayings? ')

[Document(id='5f53cc47-ab7b-4ccf-be5c-e8fa04a6d163', metadata={}, page_content='Elon Musk is a technology entrepreneur and business magnate. He is the founder, CEO, and chief engineer of SpaceX, and an early-stage investor, CEO, and product architect of Tesla, Inc. In public speeches, Elon Musk has frequently discussed the future of artificial intelligence, space exploration, and sustainable energy. He believes that AI needs careful regulation to ensure it remains safe for humanity, and he envisions colonizing Mars to make life multiplanetary.')]

#3. Augmentation

In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Load local model and tokenizer using free local GPU resources
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
)

hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.2,
    return_full_text=False
)

# Define the local HuggingFace model pipeline
llm = HuggingFacePipeline(pipeline=hf_pipe)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
prompt = PromptTemplate(
    template="""
    You are a helpful AI assistant. Use the following pieces of context to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum. Keep the answer as concise as possible.
    Always say "thanks for asking!" at the end of the answer.

    {context}

    Question : {question}
    """,
    input_variables=["context", "question"]
)



In [ ]:
question = "is the topic of elon musk present here? if yes waht was it?"
retrieved_docs = retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

final_prompt = prompt.invoke({"context" : context_text, "question" : question})

final_prompt

#step 4 - Generation

In [20]:
# Ensure the retrieved documents are combined into context
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

# Safely generate the final prompt
final_prompt = prompt.invoke({"context" : context_text, "question" : question})

# Generate the final response using our local Hugging Face pipeline
answer = llm.invoke(final_prompt)
print(answer)

NameError: name 'retrieved_docs' is not defined